In [1]:
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
from sklearn.preprocessing import LabelEncoder,OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier ,GradientBoostingClassifier
from sklearn.metrics import confusion_matrix,recall_score,r2_score,accuracy_score,precision_score,roc_auc_score,f1_score
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler


c:\Users\chira\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
train_df = pd.read_csv("../data/processed/train.csv")
test_df = pd.read_csv("../data/processed/test.csv")

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

train_df.head()

Train shape: (5634, 22)
Test shape: (1409, 22)


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,...,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,TenureGroup,ServiceCount,Churn
0,Male,0,No,No,35,No,No phone service,DSL,No,No,...,Yes,Yes,Month-to-month,No,Electronic check,49.20,1701.65,Established,3,0
1,Male,0,Yes,Yes,15,Yes,No,Fiber optic,Yes,No,...,No,No,Month-to-month,No,Mailed check,75.10,1151.55,Early,2,0
2,Male,0,Yes,Yes,13,No,No phone service,DSL,Yes,Yes,...,No,No,Two year,No,Mailed check,40.55,590.35,Early,3,0
3,Female,0,Yes,No,26,Yes,No,DSL,No,Yes,...,Yes,Yes,Two year,Yes,Credit card (automatic),73.50,1905.70,Established,5,0
4,Male,0,Yes,Yes,1,Yes,No,DSL,No,No,...,No,No,Month-to-month,No,Electronic check,44.55,44.55,New,1,0


In [3]:
X_train = train_df.drop("Churn", axis=1)
y_train = train_df["Churn"]

X_test = test_df.drop("Churn", axis=1)
y_test = test_df["Churn"]

In [4]:
print(train_df["Churn"].unique())
print(train_df["Churn"].value_counts(dropna=False))

print(test_df["Churn"].unique())
print(test_df["Churn"].value_counts(dropna=False))

[0 1]
Churn
0    4139
1    1495
Name: count, dtype: int64
[0 1]
Churn
0    1035
1     374
Name: count, dtype: int64


In [5]:
y_train = train_df["Churn"].astype(int)
y_test = test_df["Churn"].astype(int)

In [6]:
print(y_train.unique())
print(y_train.isna().sum())

print(y_test.unique())
print(y_test.isna().sum())

[0 1]
0
[0 1]
0


In [7]:
X_train = X_train.drop("customerID", axis=1, errors="ignore")
X_test = X_test.drop("customerID", axis=1, errors="ignore")

In [8]:
numerical_features = [
    "SeniorCitizen",
    "tenure",
    "MonthlyCharges",
    "TotalCharges",
    "ServiceCount"
]

In [9]:
categorical_features = [
    "gender",
    "Partner",
    "Dependents",
    "PhoneService",
    "MultipleLines",
    "InternetService",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
    "Contract",
    "PaperlessBilling",
    "PaymentMethod",
    "TenureGroup"
]

In [10]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numerical_features
        ),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            categorical_features
        )
    ]
)

In [11]:
X_train_processed = preprocessor.fit_transform(X_train)

X_test_processed = preprocessor.transform(X_test)

In [12]:
print("Before preprocessing:", X_train.shape)
print("After preprocessing:", X_train_processed.shape)

Before preprocessing: (5634, 21)
After preprocessing: (5634, 50)


In [13]:
import mlflow

mlflow.set_tracking_uri(
    "http://15.207.115.165:5000"
)

mlflow.set_experiment("telco-churn")

experiment = mlflow.get_experiment_by_name("telco-churn")

print("Experiment:", experiment.name)
print("Experiment ID:", experiment.experiment_id)
print("Tracking URI:", mlflow.get_tracking_uri())

Experiment: telco-churn
Experiment ID: 1
Tracking URI: http://15.207.115.165:5000


In [ ]:
# def evaluate_model(model, model_name, params):
    
#     with mlflow.start_run(run_name=model_name):

#         model.fit(X_train_processed, y_train)

#         y_pred = model.predict(X_test_processed)
#         y_prob = model.predict_proba(X_test_processed)[:, 1]
#         accuracy = accuracy_score(y_test, y_pred)
#         precision = precision_score(y_test, y_pred)
#         recall = recall_score(y_test, y_pred)
#         f1 = f1_score(y_test, y_pred)
#         roc_auc = roc_auc_score(y_test, y_prob)

#         mlflow.log_params(params)

#         mlflow.log_metric("accuracy", accuracy)
#         mlflow.log_metric("precision", precision)
#         mlflow.log_metric("recall", recall)
#         mlflow.log_metric("f1_score", f1)
#         mlflow.log_metric("roc_auc", roc_auc)
#         mlflow.sklearn.log_model(
#             model,
#             "model"
#         )

#         print(f"\n{model_name}")
#         print("-" * 40)
#         print(f"Accuracy : {accuracy:.4f}")
#         print(f"Precision: {precision:.4f}")
#         print(f"Recall   : {recall:.4f}")
#         print(f"F1 Score : {f1:.4f}")
#         print(f"ROC-AUC  : {roc_auc:.4f}")
        
#         return {
#             "model": model_name,
#             "accuracy": accuracy,
#             "precision": precision,
#             "recall": recall,
#             "f1_score": f1,
#             "roc_auc": roc_auc
#         }

In [24]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

import mlflow.sklearn


def evaluate_model(model, model_name, params):

    with mlflow.start_run(
        run_name=model_name,
        experiment_id=experiment.experiment_id
    ) as run:

        # Train model
        model.fit(X_train_processed, y_train)

        # Predictions
        y_pred = model.predict(X_test_processed)
        y_prob = model.predict_proba(X_test_processed)[:, 1]

        # Calculate metrics
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(
            y_test, y_pred, zero_division=0
        )
        recall = recall_score(
            y_test, y_pred, zero_division=0
        )
        f1 = f1_score(
            y_test, y_pred, zero_division=0
        )
        roc_auc = roc_auc_score(y_test, y_prob)

        # Log parameters
        mlflow.log_params(params)

        # Log metrics
        mlflow.log_metrics({
            "accuracy": accuracy,
            "precision": precision,
            "recall": recall,
            "f1_score": f1,
            "roc_auc": roc_auc
        })

        # Log model artifact
        mlflow.sklearn.log_model(
            sk_model=model,
            name="model",
            serialization_format="pickle"
        )

        print(f"\n{model_name}")
        print("-" * 40)
        print(f"Accuracy : {accuracy:.4f}")
        print(f"Precision: {precision:.4f}")
        print(f"Recall   : {recall:.4f}")
        print(f"F1 Score : {f1:.4f}")
        print(f"ROC-AUC  : {roc_auc:.4f}")
        print("Experiment ID:", run.info.experiment_id)
        print("Run ID:", run.info.run_id)

        return {
            "run_id": run.info.run_id,
            "model": model_name,
            "accuracy": accuracy,
            "precision": precision,
            "recall": recall,
            "f1_score": f1,
            "roc_auc": roc_auc
        }

In [15]:
logistic_params = {
    "C": 1.0,
    "max_iter": 1000,
    "class_weight": "balanced"
}

logistic_model = LogisticRegression(
    C=1.0,
    max_iter=1000,
    class_weight="balanced"
)

In [16]:
decision_tree_params = {
    "max_depth": 6,
    "min_samples_split": 10,
    "min_samples_leaf": 5,
    "class_weight": "balanced"
}

decision_tree_model = DecisionTreeClassifier(
    max_depth=6,
    min_samples_split=10,
    min_samples_leaf=5,
    class_weight="balanced",
    random_state=42
)

In [17]:
logistic_result = evaluate_model(
    logistic_model,
    "Logistic Regression",
    logistic_params
)

decision_tree_result = evaluate_model(
    decision_tree_model,
    "Decision Tree",
    decision_tree_params
)


Logistic Regression
----------------------------------------
Accuracy : 0.7331
Precision: 0.4983
Recall   : 0.7914
F1 Score : 0.6116
ROC-AUC  : 0.8418
Experiment ID: 1
Run ID: c4cdf2aa55304277b9af452aa484989c
🏃 View run Logistic Regression at: http://15.207.115.165:5000/#/experiments/1/runs/c4cdf2aa55304277b9af452aa484989c
🧪 View experiment at: http://15.207.115.165:5000/#/experiments/1

Decision Tree
----------------------------------------
Accuracy : 0.7388
Precision: 0.5051
Recall   : 0.7888
F1 Score : 0.6159
ROC-AUC  : 0.8280
Experiment ID: 1
Run ID: f60d63c53b3747258309200fb44f0868
🏃 View run Decision Tree at: http://15.207.115.165:5000/#/experiments/1/runs/f60d63c53b3747258309200fb44f0868
🧪 View experiment at: http://15.207.115.165:5000/#/experiments/1


In [18]:
random_forest_params = {
    "n_estimators": 200,
    "max_depth": 12,
    "min_samples_split": 10,
    "min_samples_leaf": 4,
    "class_weight": "balanced",
    "random_state": 42
}

random_forest_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=12,
    min_samples_split=10,
    min_samples_leaf=4,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

In [19]:
random_forest_result = evaluate_model(
    random_forest_model,
    "Random Forest",
    random_forest_params
)


Random Forest
----------------------------------------
Accuracy : 0.7686
Precision: 0.5482
Recall   : 0.7299
F1 Score : 0.6261
ROC-AUC  : 0.8405
Experiment ID: 1
Run ID: b335c8e302504f74b67a0e84deca3d59
🏃 View run Random Forest at: http://15.207.115.165:5000/#/experiments/1/runs/b335c8e302504f74b67a0e84deca3d59
🧪 View experiment at: http://15.207.115.165:5000/#/experiments/1


In [20]:

gradient_boosting_params = {
    "n_estimators": 150,
    "learning_rate": 0.05,
    "max_depth": 3,
    "min_samples_split": 10,
    "min_samples_leaf": 5,
    "random_state": 42
}

gradient_boosting_model = GradientBoostingClassifier(
    n_estimators=150,
    learning_rate=0.05,
    max_depth=3,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=42
)

In [21]:
gradient_boosting_result = evaluate_model(
    gradient_boosting_model,
    "Gradient Boosting",
    gradient_boosting_params
)


Gradient Boosting
----------------------------------------
Accuracy : 0.8041
Precision: 0.6701
Recall   : 0.5160
F1 Score : 0.5831
ROC-AUC  : 0.8457
Experiment ID: 1
Run ID: 98b540b0700f4dc787849d32085f0e52
🏃 View run Gradient Boosting at: http://15.207.115.165:5000/#/experiments/1/runs/98b540b0700f4dc787849d32085f0e52
🧪 View experiment at: http://15.207.115.165:5000/#/experiments/1


In [22]:
from lightgbm import LGBMClassifier

lightgbm_params = {
    "n_estimators": 300,
    "learning_rate": 0.05,
    "max_depth": -1,
    "num_leaves": 31,
    "class_weight": "balanced",
    "random_state": 42,
    "verbosity": -1
}

lightgbm_model = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=-1,
    num_leaves=31,
    class_weight="balanced",
    random_state=42,
    verbosity=-1
)

In [25]:
lightgbm_result = evaluate_model(
    lightgbm_model,
    "LightGBM",
    lightgbm_params
)

2026/10/09 17:16:26 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



LightGBM
----------------------------------------
Accuracy : 0.7608
Precision: 0.5359
Recall   : 0.7380
F1 Score : 0.6209
ROC-AUC  : 0.8326
Experiment ID: 1
Run ID: eb97b4ee3be64db79873cf3a463591cb
🏃 View run LightGBM at: http://15.207.115.165:5000/#/experiments/1/runs/eb97b4ee3be64db79873cf3a463591cb
🧪 View experiment at: http://15.207.115.165:5000/#/experiments/1


In [26]:
from xgboost import XGBClassifier

xgboost_params = {
    "n_estimators": 300,
    "max_depth": 4,
    "learning_rate": 0.05,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "eval_metric": "logloss",
    "random_state": 42,
    "n_jobs": -1
}

xgboost_model = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

In [27]:
xgboost_result = evaluate_model(
    xgboost_model,
    "XGBoost",
    xgboost_params
)

2026/10/09 17:17:48 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



XGBoost
----------------------------------------
Accuracy : 0.8034
Precision: 0.6611
Recall   : 0.5321
F1 Score : 0.5896
ROC-AUC  : 0.8421
Experiment ID: 1
Run ID: ce7990c95fcb455ea9672fb34fa28634
🏃 View run XGBoost at: http://15.207.115.165:5000/#/experiments/1/runs/ce7990c95fcb455ea9672fb34fa28634
🧪 View experiment at: http://15.207.115.165:5000/#/experiments/1


In [28]:
from sklearn.naive_bayes import GaussianNB

naive_bayes_params = {
    "var_smoothing": 1e-9
}

naive_bayes_model = GaussianNB(
    var_smoothing=1e-9
)

In [29]:
naive_bayes_result = evaluate_model(
    naive_bayes_model,
    "Gaussian Naive Bayes",
    naive_bayes_params
)

2026/10/09 17:18:25 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



Gaussian Naive Bayes
----------------------------------------
Accuracy : 0.7026
Precision: 0.4661
Recall   : 0.8262
F1 Score : 0.5959
ROC-AUC  : 0.8115
Experiment ID: 1
Run ID: 9b928e1c69094ba1ab830d357acc048d
🏃 View run Gaussian Naive Bayes at: http://15.207.115.165:5000/#/experiments/1/runs/9b928e1c69094ba1ab830d357acc048d
🧪 View experiment at: http://15.207.115.165:5000/#/experiments/1
